# Person 3 — Image feature engineering

Part of the six-person Basil Leaf ML pipeline. Run the numbered notebooks in order. This notebook states its inputs, produces a concrete handoff in `parts/artifacts`, and does not overwrite the complete project's `outputs/` results.

## Responsibility
Implement and explain the fixed image representation: RGB/HSV colour distributions, LBP texture, and HOG edge/shape descriptors. Extract exactly the same feature vector for every audited image.

**Input:** Person 2 clean manifest  
**Output:** `03_features.npz` and `03_feature_report.json`

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "Basil_Leaf_ML_Workflow.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the project folder or parts folder.")
DATA_DIR = ROOT / "data" / "raw"
ARTIFACTS = ROOT / "parts" / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SEED = 42
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
FOLDERS = {
    "Amravati_Region_Basil_Plant_Healthy": ("Healthy", 31),
    "Nagpur_Region_Basil_Plant_Healthy": ("Healthy", 473),
    "Pune_Region_Basil_Plant_Healthy": ("Healthy", 146),
    "Basil_Plant_Unhealthy": ("Unhealthy", 481),
}
print("Project:", ROOT)
print("Python:", sys.executable)
from PIL import Image, ImageOps
from skimage.color import rgb2gray, rgb2hsv
from skimage.feature import hog, local_binary_pattern
import warnings
Image.MAX_IMAGE_PIXELS=25_000_000

Project: D:\SLIIT\projectr\Dataset_Train
Python: D:\SLIIT\projectr\Dataset_Train\.venv\Scripts\python.exe


In [2]:
def read_rgb(path):
    with Image.open(path) as opened:
        opened.load(); return ImageOps.exif_transpose(opened).convert('RGB')
def handcrafted(image):
    rgb=np.asarray(image.resize((128,128),Image.Resampling.BILINEAR),dtype=np.float32)/255
    hsv=rgb2hsv(rgb); parts=[]
    for array in (rgb,hsv):
        for channel in range(3):
            hist,_=np.histogram(array[:,:,channel],bins=16,range=(0,1)); parts.append(hist.astype(np.float32)/hist.sum())
        parts.extend([array.mean(axis=(0,1)),array.std(axis=(0,1))])
    gray=(rgb2gray(rgb)*255).astype(np.uint8)
    lbp=local_binary_pattern(gray,P=8,R=1,method='uniform')
    hist,_=np.histogram(lbp,bins=np.arange(11)); parts.append(hist.astype(np.float32)/hist.sum())
    small=np.asarray(image.resize((64,64)).convert('L'),dtype=np.float32)/255
    parts.append(hog(small,orientations=9,pixels_per_cell=(8,8),cells_per_block=(2,2)))
    return np.concatenate([np.ravel(x) for x in parts]).astype(np.float32)
manifest_path=ARTIFACTS/'02_clean_manifest.csv'
if not manifest_path.exists(): raise FileNotFoundError('Run Person 2 notebook first.')
frame=pd.read_csv(manifest_path)
features=[]
for i,rel in enumerate(frame.path):
    features.append(handcrafted(read_rgb(DATA_DIR/rel)))
    if (i+1)%100==0 or i+1==len(frame): print(f'Features {i+1}/{len(frame)}')
X=np.asarray(features,dtype=np.float32)
if X.shape!=(len(frame),1882): raise AssertionError(f'Unexpected feature shape {X.shape}')
np.savez_compressed(ARTIFACTS/'03_features.npz',X=X)
report={"images":len(frame),"feature_count":X.shape[1],"colour_features":108,"texture_features":10,"hog_features":1764,"version":"rgb-hsv-lbp-hog-v1","whole_image_caveat":"Background and lighting are included because leaves are not segmented."}
(ARTIFACTS/'03_feature_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print(report)

Features 100/897


Features 200/897


Features 300/897


Features 400/897


Features 500/897


Features 600/897


Features 700/897


Features 800/897


Features 897/897


{'images': 897, 'feature_count': 1882, 'colour_features': 108, 'texture_features': 10, 'hog_features': 1764, 'version': 'rgb-hsv-lbp-hog-v1', 'whole_image_caveat': 'Background and lighting are included because leaves are not segmented.'}


## Handoff to Person 4
Supply the numeric feature matrix and feature-version report. Explain that feature extraction is fixed before cross-validation and does not use test labels.